# Experimenting with Scrapy

In [1]:
import scrapy 
import time 
import sys, json 
import socket 
import requests 
import scrapy 
from scrapy.crawler import CrawlerProcess 


import requests 
from parsel import Selector 

In [2]:
url = "https://quotes.toscrape.com/page/1/" 
html = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}).text 
sel = Selector(text=html)

print("Artile title:", sel.xpath('//title/text()').get()) 
print("Artile title v2:", sel.css("title::text").get()) 

print("Quotes:", sel.css("span.text::text").getall()) 

Artile title: Quotes to Scrape
Artile title v2: Quotes to Scrape
Quotes: ['“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”', '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”', "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”", '“Try not to become a man of success. Rather become a man of value.”', '“It is better to be hated for what you are than to be loved for what you are not.”', "“I have not failed. I've just found 10,000 ways that won't work.”", "“A woman is like a tea bag; you never know how strong it is until it's in hot water.”", '“A day without sunshine is like, you know, n

# A small test spider that gets a single page

In [3]:
import scrapy
import time 
import sys, json
import socket
import requests
import scrapy
from scrapy.crawler import CrawlerProcess

class SimpleSpider(scrapy.Spider):
    custom_settings = {
        "TWISTED_REACTOR": "twisted.internet.epollreactor.EPollReactor",
    }
    name = "my_spider"
    start_urls = ['https://quotes.toscrape.com/page/1/']  # Replace with the URL you want to fetch

    def parse(self, response):
        print("Page title:", response.xpath('//title/text()').get())
        print("Page title v2:", response.css('title::text').getall())
        print("Quotes:", response.css("span.text::text").getall())
        print("Authors:", response.css("small.author::text").getall())
        print("Tags:", response.css("a.tag::text").getall())
        for quote in response.css('div.quote'):
            print ('text:', quote.css('span.text::text').get(), 'author:', quote.css('small.author::text').get(), 'tags:', quote.css('div.tags a.tag::text').getall())
        print("Page raw html content:", response.text[:500])  # Print first 500 characters

# Run Scrapy inside Jupyter Notebook
process = CrawlerProcess(settings={
    "LOG_LEVEL": "WARN",  # Suppress logs
})
process.crawl(SimpleSpider)
process.start()

# Hungarian news spider

In [4]:
import scrapy
import time 
import sys, json
import socket
import requests
import scrapy
from scrapy.crawler import CrawlerProcess

class HungariannewsSpider(scrapy.Spider):
    custom_settings = {
        "TWISTED_REACTOR": "twisted.internet.epollreactor.EPollReactor",
    }
    name = 'telexhu'
    allowed_domains = ['telex.hu']
    start_urls = ['https://telex.hu/']

    def parse(self, response):
        #Extracting the content using xpath selectors
        title = response.css("div.title-section h1::text").extract()
        content = response.xpath("//div[@class='article-html-content']//text()").extract()


        yield {'url':response.url,'title':title,'content':content}


        #get links to follow
        for next_page in response.css("a::attr(href)").getall():
            if next_page is not None\
                    and not next_page.startswith('javascript')\
                    and not next_page.startswith('mailto'):
                next_page = response.urljoin(next_page)
                yield {'base_url':response.url, 'link':next_page}
                yield scrapy.Request(next_page, callback=self.parse)

In [5]:
import os
outfile = 'hungariannews.json'
if os.path.isfile(outfile):
    os.remove(outfile) 
    
process = CrawlerProcess(settings={
    #'LOG_LEVEL': 'WARN',  # Suppress logs, uncomment if you are ready
    'FEED_FORMAT': 'json', # jsonline is also useful
    'FEED_URI': outfile,
    'CLOSESPIDER_ITEMCOUNT': 1000, # Or stop by kernel interrupt
    'FEED_EXPORT_ENCODING': 'utf-8'
})
process.crawl(HungariannewsSpider)

process.start()

2026-03-21 11:06:46 [scrapy.utils.log] INFO: Scrapy 2.14.2 started (bot: scrapybot)
2026-03-21 11:06:46 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.2',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '25.5.0',
 'Python': '3.14.3 (main, Feb  4 2026, 01:51:49) [Clang 21.1.4 ]',
 'pyOpenSSL': '26.0.0 (OpenSSL 3.5.5 27 Jan 2026)',
 'cryptography': '46.0.5',
 'Platform': 'macOS-15.1-arm64-arm-64bit-Mach-O'}
2026-03-21 11:06:46 [scrapy.crawler] DEBUG: Using CrawlerProcess
2026-03-21 11:06:46 [scrapy.addons] INFO: Enabled addons:
[]


In [6]:
import pandas as pd
alldata=pd.read_json(outfile)
alldata

Unhandled error in Deferred:
2026-03-21 11:06:46 [twisted] CRITICAL: Unhandled error in Deferred:
2026-03-21 11:06:46 [twisted] CRITICAL: Unhandled error in Deferred:

Traceback (most recent call last):
  File "/Users/tleo/Downloads/git/school/www/.venv/lib/python3.14/site-packages/twisted/internet/defer.py", line 1853, in _inlineCallbacks
    result = context.run(
  File "/Users/tleo/Downloads/git/school/www/.venv/lib/python3.14/site-packages/twisted/python/failure.py", line 467, in throwExceptionIntoGenerator
    return g.throw(self.value.with_traceback(self.tb))
  File "/Users/tleo/Downloads/git/school/www/.venv/lib/python3.14/site-packages/scrapy/crawler.py", line 423, in _crawl
    yield d
  File "/Users/tleo/Downloads/git/school/www/.venv/lib/python3.14/site-packages/twisted/internet/defer.py", line 1857, in _inlineCallbacks
    result = context.run(gen.send, result)
  File "/Users/tleo/Downloads/git/school/www/.venv/lib/python3.14/site-packages/scrapy/crawler.py", line 152, in c

FileNotFoundError: File hungariannews.json does not exist

In [ ]:
links = alldata[alldata['base_url']==alldata['base_url']][['base_url','link']] # not NaN
links

,base_url,link
1,https://telex.hu/,https://telex.hu/
2,https://telex.hu/,https://telex.hu/after
3,https://telex.hu/,https://telex.hu/g7
4,https://telex.hu/,https://telex.hu/karakter
5,https://telex.hu/,https://telex.hu/tamogatas
...,...,...
3455,https://telex.hu/,https://www.facebook.com/telexhu
3456,https://telex.hu/,https://www.instagram.com/telexponthu/
3457,https://telex.hu/,https://www.youtube.com/c/telexponthu
3458,https://telex.hu/,https://twitter.com/telexhu


In [ ]:
news = alldata[alldata['url']==alldata['url']][['url','title','content']] # not NaN
news

,url,title,content
0,https://telex.hu/,[],[]
48,https://telex.hu/tamogatas,[],[]
69,https://telex.hu/legfrissebb,[],[]
75,https://telex.hu/karakter,[],[]
78,https://telex.hu/g7,[],[]
87,https://telex.hu/gazdasag/2026/03/09/nagy-mart...,"[Nagy Márton: Megtiltja a kormány a nyersolaj,...",[Az uniós minimumszintre csökkenti a kormány a...
112,https://telex.hu/after,[],[]
125,https://telex.hu/tamogatas/profil,[],[]
128,https://telex.hu/belfold/2026/03/09/ot-partlis...,"[Minden eddiginél kevesebb, csak öt párt listá...","[A rendszerváltás óta a legkevesebb, öt pártli..."
132,https://telex.hu/belfold/2026/03/09/buzas-habe...,[A Pécs Pride szervezőjének ügyét is felfügges...,[Hasonlóan döntött a Pécsi Járásbíróság a Prid...


# Transfermarkt spider

In [ ]:
import scrapy
import time 
import sys, json
import socket
import requests
import scrapy
from scrapy.crawler import CrawlerProcess
import datetime

class Transfermarkt(scrapy.Spider):
    custom_settings = {
        "TWISTED_REACTOR": "twisted.internet.epollreactor.EPollReactor",
    }
    name = 'transfermarkt'
    allowed_domains = ['transfermarkt.com']

    def get_club(self, link):
        if link.xpath('.//a'):
            return link.xpath('.//a//text()').extract()
        else:
            return link.xpath('.//text()').extract()

    def start_requests(self):
        start_date = datetime.date(2024, 9, 30)
        end_date = datetime.date.today()
        delta = datetime.timedelta(days=1)
        while start_date <= end_date:
            yield scrapy.Request(url='https://www.transfermarkt.com/transfers/transfertagedetail/statistik/top/plus/0/galerie/0?'\
                                     'land_id_ab=&land_id_zu=&leihe=&datum=' + start_date.strftime('%Y-%m-%d'), callback=self.parse)
            start_date += delta

    def parse(self, response):
        for line in response.xpath('//tr[@class="odd" or @class="even"]'):
            links=line.xpath('.//td[@class="hauptlink"]')
            data = {"name":links[0].xpath('.//a//text()').extract()[0], "left":self.get_club(links[1])[0], "joined":self.get_club(links[2])[0]}
            yield data
        next_page = response.xpath('//li[@class="naechste-seite"]//a/@href').extract()
        if next_page:
            yield scrapy.Request(url='https://transfermarkt.com' + next_page[0], callback=self.parse)

In [ ]:
import os
outfile = 'transfermarkt.json'
if os.path.isfile(outfile):
    os.remove(outfile) 
    
process = CrawlerProcess(settings={
    'LOG_LEVEL': 'WARN',  # Suppress logs
    'FEED_FORMAT': 'json',
    'FEED_URI': outfile,
    'FEED_EXPORT_ENCODING': 'utf-8',
    'USER_AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
})
process.crawl(Transfermarkt)

process.start()

2026-03-09 21:37:43 [py.warnings] WARNING: /home/benczur/.local/lib/python3.9/site-packages/scrapy/extensions/feedexport.py:455: ScrapyDeprecationWarning: The `FEED_URI` and `FEED_FORMAT` settings have been deprecated in favor of the `FEEDS` setting. Please see the `FEEDS` setting docs for more details
  exporter = cls(crawler)

2026-03-09 21:37:43 [py.warnings] WARNING: /home/benczur/.local/lib/python3.9/site-packages/scrapy/core/spidermw.py:433: ScrapyDeprecationWarning: __main__.Transfermarkt defines the deprecated start_requests() method. start_requests() has been deprecated in favor of a new method, start(), to support asynchronous code execution. start_requests() will stop being called in a future version of Scrapy. If you use Scrapy 2.13 or higher only, replace start_requests() with start(); note that start() is a coroutine (async def). If you need to maintain compatibility with lower Scrapy versions, when overriding start_requests() in a spider class, override start() as well; 

In [ ]:
import pandas as pd
transferdata=pd.read_json(outfile)
transferdata

RuntimeError: module was compiled against NumPy C-API version 0x10 (NumPy 1.23) but the running NumPy has C-API version 0xf. Check the section C-API incompatibility at the Troubleshooting ImportError section at https://numpy.org/devdocs/user/troubleshooting-importerror.html#c-api-incompatibility for indications on how to solve this problem.

,name,left,joined
0,Sofiane Boudraa,Valenciennes FC,Aubagne Air Bel
1,Abdo Gouda,Pyramids FC,Harras Hodoud
2,Corry Evans,Without Club,Bradford
3,Seidu Basit,Al-Hilal,Fujairah SC
4,Gabriel Torje,Concordia,Campulung
...,...,...,...
10547,Yubo Hu,HM Codion,LZ Longyuan At.
10548,Andrius Kaulinis,Riteriai,Hegelmann
10549,Vladislav Ignatjev,BATE Borisov,Dnepr Mogilev
10550,Håvar Jenssen,Sarpsborg 08,KFUM
